# Implement pure version of SCC cycle

## Imports

In [1]:
import torch
from typing import Optional, Dict, Any, Literal, Union
from torch import Tensor
import xitorch as xt
from xitorch.optimize.rootfinder import equilibrium
from torchviz import make_dot
from tbmalt import OrbitalInfo
from tbmalt import Geometry

### Used functions import

In [2]:
from tbmalt.common.maths import eighb
from tbmalt.common.batch import prepeat_interleave
from tbmalt.physics.filling import (
    fermi_search, fermi_smearing, gaussian_smearing, entropy_term,
    aufbau_filling)

In [3]:
torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=16)

## Functions used in the scc cycle

### Prepeat interleave for shift calc

In [4]:
# def prepeat_interleave(tensor: Tensor, repeats: Tensor, value=0):
#     if tensor.ndim <= 1:
#         return tensor.repeat_interleave(repeats)
#     else:
#         assert tensor.shape == repeats.shape
#         return pack([i.repeat_interleave(j) for i, j in
#                      zip(tensor, repeats)], value=value)

### Code for occupancy

In [5]:
# def _middle_gap_approximation(
#         eigenvalues: Tensor, n_electrons: Tensor, scale_factor: Tensor,
#         e_mask: Optional[Tensor] = None, return_occupations: bool = False
#         ) -> Tuple[Tensor, Tensor]:
#     """Returns the midpoint between the HOMO and LUMO."""

#     # Shape of Ɛ tensor where k-points & spin-channels have been flattened out.
#     # Note that only spin-channels with common fermi energies get flattened.
#     shape = torch.Size([*n_electrons.shape, -1])

#     # Flatten & sort the eigenvalues so there's 1 row per-system; the
#     # spin dimension is only flattened when the spin channels share a
#     # common fermi-energy.
#     eigenvalues_flat, srt = psort(
#         eigenvalues.view(shape),
#         None if e_mask is None else e_mask.view(shape))

#     # Maximum occupation of each eigenstate, sorted and flattened.
#     occupations = (torch.ones_like(eigenvalues_flat) * scale_factor
#                    ).gather(-1, srt)

#     # Locate HOMO index, via the transition between under/over filled.
#     # An indirect method is used here as direct calls to ">" & "<" result in
#     # spurious behaviour when any noise is present in `n_electrons`.
#     occupations_cs = bT(occupations.cumsum(-1)) - n_electrons
#     r = torch.finfo(n_electrons.dtype).resolution * 5
#     i_homo = torch.argmax(
#         torch.as_tensor(bT(occupations_cs.ge(-r)), dtype=torch.long),
#         dim=-1).view(shape)

#     # Identify the index of the LUMO. Care must be taken to catch the case
#     # where i_lumo is out of bounds due to the LUMO state not being present in
#     # the eigenvalues. This is encountered when all states are fully occupied.
#     if e_mask is not None:
#         n_states = e_mask.view(shape).sum(dim=-1).view(i_homo.shape)
#     else:
#         n_states = torch.tensor(eigenvalues_flat.shape[-1])

#     i_lumo = torch.minimum(i_homo + 1, n_states - 1)

#     mid_point = eigenvalues_flat.gather(
#         -1, torch.cat((i_homo, i_lumo), -1)).mean(-1)

#     # Return the mid-point
#     if not return_occupations:
#         return mid_point
#     # Unless also instructed to also return the occupations
#     else:
#         # Set the occupancies of all states above the HOMO equal to zero
#         for n, i in enumerate(i_homo.flatten()):
#             torch.atleast_2d(occupations)[n, i + 1:] = 0.0

#         # Re-order and un-flatten the occupancy array to match its original form
#         occupations = occupations.gather(
#             -1, torch.argsort(srt, -1)).view(eigenvalues.shape)

#         return mid_point, occupations

In [6]:
# def aufbau_filling(
#         eigenvalues: Tensor, n_electrons: float_like,
#         e_mask: Optional[Union[Tensor, OrbitalInfo]] = None,
#         k_weights: Optional[Tensor] = None) -> Tensor:
#     # No comments are provided for the code here as all the code present is
#     # functionally identical to that in `fermi_search`; albeit a much cut down
#     # version.

#     if not isinstance(n_electrons, Tensor) \
#             or not torch.is_floating_point(n_electrons):
#         n_electrons = torch.as_tensor(
#             n_electrons, dtype=eigenvalues.dtype, device=eigenvalues.device)

#     if isinstance(e_mask, OrbitalInfo):
#         e_mask = e_mask.on_atoms != -1

#     pf = 5 - eigenvalues.ndim - [k_weights, e_mask].count(None)
#     scale_factor = pf if k_weights is None else pf * k_weights

#     shp = torch.Size([*n_electrons.shape, -1])

#     if torch.lt(n_electrons.abs(), torch.finfo(eigenvalues.dtype).eps).any():
#         raise ValueError('Number of elections cannot be zero.')

#     if torch.any((n_electrons / pf).gt(
#             eigenvalues.view(shp).shape[-1] if e_mask is None
#             else e_mask.view(shp).count_nonzero(-1))):

#         raise ValueError('Number of electrons cannot exceed 2 * n states')

#     # Divide by the pre-factor to get back to fractional values. This is a bit
#     # wasteful and thus the occupancy scaling situation should be refactored
#     # at some point in the future.
#     return _middle_gap_approximation(
#         eigenvalues, n_electrons, scale_factor,
#         e_mask, return_occupations=True)[1] / pf

In [7]:
# def occupancy(self):
#     """Occupancies of each state"""

#     # Note that this scale factor assumes spin-restricted and will need to
#     # be refactored when implementing spin-unrestricted calculations.
#     scale_factor = 2.0

#     # If finite temperature is active then use the appropriate smearing
#     # method.
#     return aufbau_filling(
#         self.eig_values, self.n_electrons,
#         e_mask=self.orbs if self.is_batch else None) * scale_factor


In [8]:
def occupancy_pure(eig_values, n_electrons):
    """Occupancies of each state"""

    # Note that this scale factor assumes spin-restricted and will need to
    # be refactored when implementing spin-unrestricted calculations.
    scale_factor = 2.0

    # If finite temperature is active then use the appropriate smearing
    # method.
    return aufbau_filling(
        eig_values, n_electrons, None) * scale_factor


### Eigh solver

In [9]:
# def eighb(a: Tensor,
#           b: Tensor = None,
#           scheme: Literal['chol', 'lowd'] = 'chol',
#           broadening_method: Optional[Literal['cond', 'lorn']] = 'cond',
#           factor: float = 1E-12,
#           sort_out: bool = True,
#           aux: bool = True,
#           **kwargs) -> Tuple[Tensor, Tensor]:
#     # Initial setup to make function calls easier to deal with
#     # If smearing use _SymEigB otherwise use torch.linalg.eigh
#     func = _SymEigB.apply if broadening_method else torch.linalg.eigh
#     # Set up for the arguments
#     args = (broadening_method, factor) if broadening_method else ()

#     if aux:
#         is_zero = torch.eq(a, 0)
#         mask = torch.all(is_zero, dim=-1) & torch.all(is_zero, dim=-2)

#     if b is None:  # For standard eigenvalue problem
#         if aux:
#             # Convert from zero-padding to padding with largest eigenvalue estimate
#             shift = estimate_minmax(a)[-1].unsqueeze(-1)
#             a = a + torch.diag_embed(shift * mask)

#         w, v = func(a, *args)  # Call the required eigen-solver

#     else:  # Otherwise it will be a general eigenvalue problem

#         # Cholesky decomposition can only act on positive definite matrices;
#         # which is problematic for zero-padded tensors. Similar issues are
#         # encountered in the Löwdin scheme. To ensure positive definiteness
#         # the diagonals of padding columns/rows are therefore set to 1.

#         # Create a mask which is True wherever a column/row pair is 0-valued
#         is_zero = torch.eq(b, 0)
#         mask = torch.all(is_zero, dim=-1) & torch.all(is_zero, dim=-2)

#         # Set the diagonals at these locations to 1
#         b = b + torch.diag_embed(mask.type(a.dtype))

#         # For Cholesky decomposition scheme
#         if scheme == 'chol':

#             # Perform Cholesky factorization (A = LL^{T}) of B to attain L
#             l = torch.linalg.cholesky(b)

#             # Compute the inverse of L:
#             if kwargs.get('direct_inv', False):
#                 # Via the direct method if specifically requested
#                 l_inv = torch.inverse(l)
#             else:
#                 # Otherwise compute via an indirect method (default)
#                 identity = torch.zeros_like(l)
#                 identity.diagonal(dim1=-2, dim2=-1)[:] = 1
#                 l_inv = torch.linalg.solve(l, identity)

#             # Transpose of l_inv: improves speed in batch mode
#             l_inv_t = torch.transpose(l_inv, -1, -2)

#             # To obtain C, perform the reduction operation C = L^{-1}AL^{-T}
#             c = l_inv @ a @ l_inv_t

#             if aux:
#                 # Convert from zero-padding to padding with largest eigenvalue estimate
#                 shift = estimate_minmax(c)[-1].unsqueeze(-1)
#                 c = c + torch.diag_embed(shift * mask)

#             # The eigenvalues of Az = λBz are the same as Cy = λy; hence:
#             w, v_ = func(c, *args)

#             # Eigenvectors, however, are not, so they must be recovered:
#             #   z = L^{-T}y
#             v = l_inv_t @ v_

#         elif scheme == 'lowd':  # For Löwdin Orthogonalisation scheme

#             # Perform the BV = WV eigen decomposition.
#             w, v = func(b, *args)

#             # Embed w to construct "small b"; inverse power is also done here
#             # to avoid inf values later on.
#             b_small = torch.diag_embed(w ** -0.5)

#             # Construct symmetric orthogonalisation matrix via:
#             #   B^{-1/2} = V b^{-1/2} V^{T}
#             b_so = v @ b_small @ v.transpose(-1, -2)

#             # A' (a_prime) can then be constructed as: A' = B^{-1/2} A B^{-1/2}
#             a_prime = b_so @ a @ b_so

#             if aux:
#                 # Convert from zero-padding to padding with largest eigenvalue estimate
#                 shift = estimate_minmax(a_prime)[-1].unsqueeze(-1)
#                 a_prime = a_prime + torch.diag_embed(shift * mask)

#             # Decompose the now orthogonalised A' matrix
#             w, v_prime = func(a_prime, *args)

#             # the correct eigenvector is then recovered via
#             #   V = B^{-1/2} V'
#             v = b_so @ v_prime

#         else:  # If an unknown scheme was specified
#             raise ValueError('Unknown scheme selected.')

#     # If sort_out is enabled, nullify the "ghost" eigen-values
#     if sort_out:
#         if aux:
#             w = torch.where(~mask, w, w.new_tensor(0))
#         else:
#             w, v = _eig_sort_out(w, v, not aux)

#     # Return the eigenvalues and eigenvectors
#     return w, v


### Mulliken

In [10]:
def _mulliken(
        rho: Tensor, S: Tensor, orbs: Optional[OrbitalInfo] = None,
        resolution: Optional[Literal['atom', 'shell', 'orbital']] = None
) -> Tensor:
    if resolution is not None and orbs is None:
        raise TypeError(
            '"resolution" overrides default behaviour associated with the '
            '"orbs" object\'s "shell_resolved" attribute. Thus it cannot be '
            'specified in absence of the "orbs" argument.')

    q = (rho * S).sum(-1)  # Calculate the per-orbital Mulliken populations
    if orbs is not None:  # Resolve to per-shell/atom if instructed to
        if resolution is None:
            # TODO: Change orbs to have a res_matrix_shape property.
            size, ind = orbs.res_matrix_shape, orbs.on_res
        elif resolution == 'atom':
            size, ind = orbs.atomic_matrix_shape, orbs.on_atoms
        elif resolution == 'shell':
            size, ind = orbs.shell_matrix_shape, orbs.on_shells
        else:
            raise NotImplementedError("Unknown resolution")

        q = torch.zeros(size[:-1], device=rho.device, dtype=rho.dtype
                        ).scatter_add_(-1, ind.clamp(min=0), q)

    return q

## Input parameters

In [11]:
q_zero_res = torch.tensor([5.999999999999999, 1.000000000000000, 1.000000000000000], requires_grad = False)

In [12]:
gamma = torch.tensor([[0.495404170221900, 0.375434274999402, 0.375434274999402],
        [0.375434274999402, 0.419617426124700, 0.283520366296857],
        [0.375434274999402, 0.283520366296857, 0.419617426124700]], requires_grad = False)

In [13]:
orbs_per_res = None

In [14]:
core_hamiltonian = torch.tensor([[-0.878832584077500,  0.000000000000000,  0.000000000000000,  0.000000000000000, -0.521823668650510, -0.521823668650510],
        [ 0.000000000000000, -0.332131773529400,  0.000000000000000,  0.000000000000000, -0.300109606072966,  0.300109606072966],
        [ 0.000000000000000,  0.000000000000000, -0.332131773529400,  0.000000000000000,  0.187568503795603,  0.187568503795603],
        [ 0.000000000000000,  0.000000000000000,  0.000000000000000, -0.332131773529400,  0.000000000000000,  0.000000000000000],
        [-0.521823668650510, -0.300109606072966,  0.187568503795603,  0.000000000000000, -0.238600544048200, -0.099647452215670],
        [-0.521823668650510,  0.300109606072966,  0.187568503795603,  0.000000000000000, -0.099647452215670, -0.238600544048200]], requires_grad = False)

In [15]:
overlap = torch.tensor([[ 1.000000000000000,  0.000000000000000,  0.000000000000000,  0.000000000000000,  0.442488678677807,  0.442488678677807],
        [ 0.000000000000000,  1.000000000000000,  0.000000000000000,  0.000000000000000,  0.340943838131014, -0.340943838131014],
        [ 0.000000000000000,  0.000000000000000,  1.000000000000000,  0.000000000000000, -0.213089898831884, -0.213089898831884],
        [ 0.000000000000000,  0.000000000000000,  0.000000000000000,  1.000000000000000,  0.000000000000000,  0.000000000000000],
        [ 0.442488678677807,  0.340943838131014, -0.213089898831884,  0.000000000000000,  1.000000000000000,  0.164075254022370],
        [ 0.442488678677807, -0.340943838131014, -0.213089898831884,  0.000000000000000,  0.164075254022370,  1.000000000000000]], requires_grad = True)

In [16]:
orbs = None

In [17]:
n_electrons = torch.tensor([8])#MAYBE WRONG

### Implement geo and orbs

In [18]:
H2O = Geometry(torch.tensor([8, 1, 1]), 
               torch.tensor([[0.0, 0.0, 0.0],
                             [0.0, 0.8, -0.5],
                             [0.0, -0.8, -0.5]], requires_grad=True),
               units='angstrom'
               )
shell_dict = {1: [0], 6: [0,1], 8: [0, 1], 16: [0, 1, 2], 79: [0, 1, 2]}
orbs = OrbitalInfo(H2O.atomic_numbers, shell_dict, shell_resolved=False)

## Function implementation

In [19]:
def _scc_cycle(q_in, 
               q_zero_res, 
               gamma, 
               orbs,
               core_hamiltonian,
               overlap,
               n_electrons
              ) -> Tensor:

    # Construct the shift matrix
    shifts = torch.einsum(
        '...i,...ij->...j', q_in - q_zero_res, gamma)
    shifts = prepeat_interleave(shifts, orbs.orbs_per_res)
    shifts = (shifts[..., None] + shifts[..., None, :])

    # Compute the second order Hamiltonian matrix
    hamiltonian = core_hamiltonian + .5 * overlap * shifts
    #print(hamiltonian)

    # Obtain the eigen-values/vectors via an eigen decomposition
    eig_values, eig_vectors = eighb(
        hamiltonian, overlap)
    #print(eig_values)

    occupancy = occupancy_pure(eig_values, n_electrons)
    # Scaled occupancy values
    s_occs = torch.einsum(
        '...i,...ji->...ji', torch.sqrt(occupancy), eig_vectors)

    # Density matrix
    rho = s_occs @ s_occs.transpose(-1, -2).conj()

    # Compute and return the new
    return _mulliken(rho, overlap, orbs)

In [20]:
q_in = torch.tensor([6.609764090040819, 0.695117954979599, 0.695117954979598], requires_grad=True)

In [21]:
res = _scc_cycle(q_in,
           q_zero_res,
           gamma,
           orbs,
           core_hamiltonian,
           overlap,
           n_electrons
          )

In [22]:
make_dot(res).render("scc_pure", format="png")

'scc_pure.png'

In [23]:
torch.autograd.grad(res, q_in, torch.ones_like(res))

(tensor([-1.2545994243426563e-16, -1.7697663226404000e-17,
         -3.2217343258175615e-17]),)

In [24]:
q0 = torch.zeros_like(q_in)
q0

tensor([0., 0., 0.])

In [25]:
para = (q_zero_res,
        gamma,
        orbs,
       core_hamiltonian,
       overlap,
       n_electrons
       )

In [26]:
res_xit = equilibrium(_scc_cycle, q0, params=para)

In [27]:
res_xit

tensor([6.6097640959048896, 0.6951178959090170, 0.6951178959090167],
       grad_fn=<_RootFinderBackward>)

In [28]:
make_dot(res_xit).render("scc_pure_xit", format="png")

'scc_pure_xit.png'

In [29]:
torch.autograd.grad(res, overlap, torch.ones_like(res))

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
torch.autograd.grad(res_xit, overlap, torch.ones_like(res))